In [1]:
# Import required libraries
import json
import pandas as pd
import os

In [4]:
# Mount Google Drive in Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
main_folder = "/content/drive/MyDrive/fhir"

In [6]:
# Empty list to store all COVID case records from all hospitals
covid_records = []

In [7]:
# Loop through each hospital folder
for hospital_folder in os.listdir(main_folder):
    hospital_path = os.path.join(main_folder, hospital_folder)

    # Make sure it is actually a folder
    if os.path.isdir(hospital_path):

        # Loop through each JSON file inside the hospital folder
        for file_name in os.listdir(hospital_path):
            if file_name.endswith(".json"):
                file_path = os.path.join(hospital_path, file_name)

                # Open and read the JSON file
                with open(file_path, "r") as file:
                    data = json.load(file)

                # FHIR bundles store resources inside "entry"
                for entry in data.get("entry", []):
                    resource = entry.get("resource", {})

                    # Only look at Condition resources (diagnoses)
                    if resource.get("resourceType") == "Condition":

                        # Get diagnosis text
                        condition_text = resource.get("code", {}).get("text", "Unknown")

                        # Filter COVID / Coronavirus cases
                        if "COVID" in condition_text.upper() or "CORONAVIRUS" in condition_text.upper():

                            # Extract useful fields
                            patient_id = resource.get("subject", {}).get("reference", "Unknown")
                            onset_date = resource.get("onsetDateTime", "Unknown")
                            encounter = resource.get("encounter", {}).get("reference", "Unknown")

                            # Save into list
                            covid_records.append({
                                "Hospital": hospital_folder,
                                "JSON_File": file_name,
                                "Patient_ID": patient_id,
                                "Condition": condition_text,
                                "Onset_Date": onset_date,
                                "Encounter": encounter
                            })

In [8]:
# Convert results to a DataFrame
df = pd.DataFrame(covid_records)

# Show summary
print("Total COVID Cases Found:", len(df))
print("\nCases by Hospital:")
print(df["Hospital"].value_counts())

Total COVID Cases Found: 275

Cases by Hospital:
Hospital
blessing-mgh-192.168.17.93               233
blessing-bcmh-192.168.17.92               26
blessing-portage-health-192.168.17.91     10
blessing-aspirus-192.168.17.90             6
Name: count, dtype: int64


In [9]:
# Save as CSV for Google Looker Studio
df.to_csv("/content/aggregated_covid_data.csv", index=False)

print("\nAggregation complete.")
print("CSV saved as /content/aggregated_covid_data.csv")


Aggregation complete.
CSV saved as /content/aggregated_covid_data.csv
